# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` fields.

### Dataset Source
The dataset source is defined via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

We'll use this URL to load the dataset's metadata, record sets, and perform data analysis.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Let's verify the dataset name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
Let's inspect the available **record sets** and their fields by their `@id` fields.

**Note:** All further steps will reference entities using their `@id` fields for full reproducibility.

In [ ]:
# List all top-level record sets (by @id)
record_sets = dataset.record_sets

print("Available record sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'Unnamed')}")

# For each record set, list fields and field @ids
for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    if not fields:
        print("  (No fields defined)")
    else:
        for fld in fields:
            print(f"  - Field @id: {fld['@id']}, name: {fld.get('name', '[no name]')}, type: {fld.get('dataType', '[unknown]')}")

## 3. Data Extraction
Let's extract tabular data from record sets into DataFrames. We'll use the `@id` of the main record set for the clinical records.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        # Load all records into DataFrame
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set: {record_set_id}")
            print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
            print(f"Sample data:\n{dataframes[record_set_id].head()}\n")
        else:
            print(f"No records found for record set: {record_set_id}\n")
    except Exception as e:
        print(f"Error for record set {record_set_id}: {e}\n")

# We'll use the main record set for clinical tabular records
# Replace with the actual @id of the primary record set after inspection above
# (Here, we pick the first with data)
main_record_set_id = None
for k, v in dataframes.items():
    if (v is not None) and (not v.empty):
        main_record_set_id = k
        break

if main_record_set_id is None:
    raise ValueError("No record set with tabular data found.")

# Show the columns and preview data for the main record set
print(f"\nMain tabular record set: {main_record_set_id}")
print("Columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll process the main DataFrame by referencing columns via their field `@id`s. We will:
- Select a numeric field for basic filtering and normalization.
- Demonstrate filtering, normalization, and grouping by another field (all via `@id`).

In [ ]:
# List columns (all are @id references)
df = dataframes[main_record_set_id]
print("Columns (@id):", df.columns.tolist())

# Choose a numeric field @id for exploration.
# We'll pick one that resembles a numeric variable (for this example, look for 'Age' or similar)
import re
numeric_field_id = None
for col in df.columns:
    if re.search('age', col, re.I):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # fallback: try to detect numeric columns heuristically
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id is None:
    raise ValueError('No numeric field found for analysis.')

print(f"Chosen numeric field @id: {numeric_field_id}")

# Filter the DataFrame for records where age > 50 (as example threshold)
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field (such as 'Sex' or similar categorical variable)
group_field_id = None
for col in df.columns:
    if re.search('sex|gender', col, re.I):
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df)
else:
    print("\nNo suitable group field (e.g., sex/gender) found for grouping.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and compare groups if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot grouped by group_field_id (if group field exists)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print('No suitable group field found for boxplot.')

## 6. Conclusion
In this notebook, we:
- Loaded and inspected the FAIR^2 dataset using the `mlcroissant` library by referencing all entities via `@id`.
- Explored record sets and their fields, extracted the main record set as a DataFrame, and performed basic filtering and normalization of a numeric variable (e.g., age at diagnosis).
- Visualized the distribution of the numeric field and compared groups by a categorical field if available.

This process can be extended to other variables, record sets, and further statistical analysis as needed for clinicopathological research.